# Quickstart: do data lake ao MLflow

Este notebook mostra o caminho completo de um modelo preditivo dentro da plataforma:

1. ler dados **silver/gold** do data lake (MinIO) com DuckDB;
2. treinar um modelo com scikit-learn;
3. registrar parâmetros, métricas e o modelo no **MLflow** (persistidos em volumes Docker);
4. carregar o modelo de volta a partir do Model Registry.

**Pré-requisito:** execute a DAG `example_api_to_bronze` no Airflow (http://localhost:8080) e aguarde a `dbt_medallion` terminar. Isso cria os dados de exemplo usados aqui.

Nenhuma senha é digitada: as credenciais do lake vêm do cofre (`secrets/`) e o endereço do MLflow vem da variável `MLFLOW_TRACKING_URI`.

In [ ]:
import mlflow
import pandas as pd

from lakehouse import LakeConfig, connect

lake = LakeConfig.from_env()   # bucket names and credentials come from the vault
con = connect()                # in-memory DuckDB already wired to the data lake

print('MLflow tracking server:', mlflow.get_tracking_uri())
print('Gold datasets available in the lake:')
con.sql(f"SELECT file FROM glob('{lake.uri('gold')}/*.parquet')").show()

## 1. Ler dados do lake

O DuckDB lê o Parquet direto do MinIO (`s3://silver/...`) e devolve um DataFrame do pandas. Para tabelas grandes, filtre e agregue no SQL antes de chamar `.df()` — o processamento acontece no DuckDB, não na memória do pandas.

In [ ]:
hourly = con.sql(f"""
    SELECT location, observed_at, temperature_c, relative_humidity_pct, precipitation_mm, wind_speed_kmh
    FROM read_parquet('{lake.uri('silver', 'slv_example_weather_hourly.parquet')}')
    ORDER BY location, observed_at
""").df()

print(hourly.shape)
hourly.tail()

## 2. Preparar as features

Objetivo do exemplo: prever a temperatura de uma hora a partir da hora do dia, da umidade, do vento e das temperaturas anteriores (1 h e 24 h antes).

In [ ]:
# Feature engineering: hour of the day and lagged temperatures per location.
data = hourly.dropna().sort_values(['location', 'observed_at']).copy()
data['hour'] = data['observed_at'].dt.hour.astype('float64')
data['temp_lag_1h'] = data.groupby('location')['temperature_c'].shift(1)
data['temp_lag_24h'] = data.groupby('location')['temperature_c'].shift(24)
data = data.dropna().sort_values('observed_at').reset_index(drop=True)

features = ['hour', 'relative_humidity_pct', 'wind_speed_kmh', 'temp_lag_1h', 'temp_lag_24h']
target = 'temperature_c'

# Chronological split: train on the past, evaluate on the most recent hours.
split = int(len(data) * 0.8)
train, test = data.iloc[:split], data.iloc[split:]
print(len(train), 'training rows /', len(test), 'test rows')

## 3. Treinar e registrar no MLflow

Tudo o que é registrado dentro de `mlflow.start_run()` fica rastreável em http://localhost:5000:

- **parâmetros e métricas** → banco Postgres (volume `postgres-data`);
- **arquivo do modelo** → bucket `mlflow` do MinIO (volume `minio-data`).

`registered_model_name` cria uma nova **versão** no Model Registry a cada execução, o que permite comparar modelos.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

mlflow.set_experiment('example_weather_temperature')

params = {'n_estimators': 200, 'max_depth': 8, 'random_state': 42}

with mlflow.start_run(run_name='random_forest_baseline') as run:
    model = RandomForestRegressor(**params).fit(train[features], train[target])
    predictions = model.predict(test[features])

    metrics = {
        'mae': float(mean_absolute_error(test[target], predictions)),
        'r2': float(r2_score(test[target], predictions)),
    }

    # Everything logged here is versioned and comparable in the MLflow UI.
    mlflow.log_params(params)
    mlflow.log_metrics(metrics)
    mlflow.set_tag('training_data', lake.uri('silver', 'slv_example_weather_hourly.parquet'))
    mlflow.sklearn.log_model(
        model,
        name='model',
        input_example=train[features].head(5),
        registered_model_name='example_weather_temperature',
    )

print('run_id:', run.info.run_id)
print(metrics)

## 4. Carregar o modelo do Model Registry

O modelo é recuperado pelo nome e versão — sem depender do notebook nem do container que o treinou.

In [ ]:
loaded = mlflow.pyfunc.load_model('models:/example_weather_temperature/latest')

sample = test[features].head(5)
pd.DataFrame({
    'actual': test[target].head(5).to_numpy(),
    'predicted': loaded.predict(sample),
})

## Próximos passos

- Troque a leitura de exemplo pelos seus datasets `gold` (`lake.uri('gold', 'meu_modelo.parquet')`).
- Para consultar o MySQL de origem direto do notebook: `con = connect(mysql=True)` e depois `con.sql("SELECT * FROM mysql_src.<banco>.<tabela> LIMIT 10")`.
- Tokens de APIs: `from lakehouse import setting; setting('API_MINHA_CHAVE', required=True)`.
- Bibliotecas extras (xgboost, lightgbm...): adicione em `docker/jupyter/requirements.txt` e rode `docker compose build jupyter && docker compose up -d jupyter`.